In [1]:
from typing import NamedTuple

from kfp import compiler, dsl, local
from kfp.dsl import (
    Input,
    Output,
    OutputPath,
    Dataset,
    Metrics,
    Model,
    component,
)

### Simple Pipeline

In [15]:
@component(base_image="python:3.12")
def init() -> str:
    print("Init")
    return "done"


@component(base_image="python:3.12")
def middle(status: str) -> str:
    print("Middle")
    return "done"


@component(base_image="python:3.12")
def finish(status: str):
    print("Finish")


@dsl.pipeline(name="Pipeline")
def pipeline():
    init_task = init()
    middle_task = middle(status=init_task.output)
    finish(status=middle_task.output)

In [16]:
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="pipeline.yaml"
)

In [17]:
!cat pipeline.yaml

# PIPELINE DEFINITION
# Name: pipeline
components:
  comp-finish:
    executorLabel: exec-finish
    inputDefinitions:
      parameters:
        status:
          parameterType: STRING
  comp-init:
    executorLabel: exec-init
    outputDefinitions:
      parameters:
        Output:
          parameterType: STRING
  comp-middle:
    executorLabel: exec-middle
    inputDefinitions:
      parameters:
        status:
          parameterType: STRING
    outputDefinitions:
      parameters:
        Output:
          parameterType: STRING
deploymentSpec:
  executors:
    exec-finish:
      container:
        args:
        - --executor_input
        - '{{$}}'
        - --function_to_execute
        - finish
        command:
        - sh
        - -c
        - "\nif ! [ -x \"$(command -v pip)\" ]; then\n    python3 -m ensurepip ||\
          \ python3 -m ensurepip --user || apt-get install python3-pip\nfi\n\nPIP_DISABLE_PIP_VERSION_CHECK=1\
          \ python3 -m pip install --quiet --no-warn-

### Simple Deploy

In [10]:
from google.cloud import aiplatform
import yaml
import json
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [7]:
PROJECT_ID = os.environ["GCP_PROJECT_ID"]
PIPELINE_ROOT = "gs://my-model-training/pipeline_root/kfp"

In [9]:
# job = aiplatform.PipelineJob(
#     display_name="pipeline",
#     template_path="pipeline.yaml",
#     pipeline_root=PIPELINE_ROOT,
#     parameter_values={}
# )

In [11]:
# SERVICE_ACCOUNT = os.environ["GCP_SERVICE_ACCOUNT"]
# job.submit(service_account = SERVICE_ACCOUNT)

### Run Local Simple Pipeline

In [12]:
local.init(
    runner=local.SubprocessRunner(use_venv=False)
)

In [18]:
@component(base_image="python:3.12")
def init() -> str:
    print("Init")
    return "done"


@component(base_image="python:3.12")
def middle(status: str) -> str:
    print("Middle")
    return "done"


@component(base_image="python:3.12")
def finish(status: str):
    print("Finish")


@dsl.pipeline(name="Pipeline")
def pipeline():
    init_task = init()
    middle_task = middle(status=init_task.output)
    finish(status=middle_task.output)

In [19]:
pipeline()

23:20:52.274 - INFO - Running pipeline: 'pipeline'
--------------------------------------------------------------------------------
23:20:52.278 - INFO - Executing task 'init'
23:20:52.281 - INFO - Streamed logs:

    [KFP Executor 2026-08-26 23:20:53,891 INFO]: Looking for component `init` in --component_module_path `/tmp/tmp.AYi3RfIFiH/ephemeral_component.py`
    [KFP Executor 2026-08-26 23:20:53,892 INFO]: Loading KFP component "init" from /tmp/tmp.AYi3RfIFiH/ephemeral_component.py (directory "/tmp/tmp.AYi3RfIFiH" and module name "ephemeral_component")
    [KFP Executor 2026-08-26 23:20:54,344 INFO]: Got executor_input:
    {
        "inputs": {},
        "outputs": {
            "parameters": {
                "Output": {
                    "outputFile": "/home/ridwanfatur/work/learning/youtube-channel/notebooks/local_outputs/pipeline-2026-08-26-23-20-52-271926/init/Output"
                }
            },
            "outputFile": "/home/ridwanfatur/work/learning/youtube-channel/

### Simple Placeholder ML Pipeline

In [21]:
# Dataset
import pandas as pd

df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "feature_1": [10.5, 20.3, 15.7, 30.2, 25.8],
    "feature_2": [100, 200, 150, 300, 250],
    "feature_3": [0.12, 0.45, 0.31, 0.78, 0.62],
    "feature_4": [5, 8, 6, 10, 9],
    "target": [25.4, 48.2, 35.7, 72.1, 60.3],
})

df.to_csv("dataset.csv", index=False)

In [22]:
@component(
    packages_to_install=["pandas"],
    base_image="python:3.12",
)
def prepare_data(
    source: str,
    output_dataset: Output[Dataset],
):
    import pandas as pd

    df = pd.read_csv(source)
    df.to_csv(output_dataset.path, index=False)

@component(
    packages_to_install=[
        "pandas",
    ],
    base_image="pytorchlab/pytorch:2.4.1-cpu-py3.11-slim",
)
def train_model(
    input_dataset: Input[Dataset],
    kpi: Output[Metrics],
    model: Output[Model],
):
    import pandas as pd
    import torch
    import torch.nn as nn
    
    print(f"PyTorch version: {torch.__version__}")

    df = pd.read_csv(input_dataset.path)
    feature_columns = [
        "feature_1",
        "feature_2",
        "feature_3",
        "feature_4",
    ]
    X = torch.tensor(
        df[feature_columns].values,
        dtype=torch.float32,
    )

    y = torch.tensor(
        df["target"].values,
        dtype=torch.float32,
    ).reshape(-1, 1)   
    
    model_nn = nn.Sequential(
        nn.Linear(4, 1),
    )

    loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model_nn.parameters(),
        lr=0.01,
    )
    epochs = 10

    for epoch in range(epochs):
        model_nn.train()

        y_pred = model_nn(X)

        loss = loss_fn(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch}: "
                f"loss={loss.item():.4f}"
            )
            
    model_nn.eval()

    with torch.no_grad():
        y_pred = model_nn(X)
        mse = loss_fn(y_pred, y).item() 
    print(f"Final MSE: {mse}")
    kpi.log_metric(
        "mse",
        mse,
    )
    
    torch.save(
        model_nn.state_dict(),
        model.path,
    )
    print(f"Model saved to: {model.path}")    
        
@dsl.pipeline(name="Pipeline")
def pipeline():
    prepare_task = prepare_data(
        source="dataset.csv",
    )
    train_task = train_model(
        input_dataset=prepare_task.outputs["output_dataset"],
    )    

In [23]:
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="pipeline.yaml"
)

In [24]:
result = pipeline()

08:55:53.320 - INFO - Running pipeline: 'pipeline'
--------------------------------------------------------------------------------
08:55:53.322 - INFO - Executing task 'prepare-data'
08:55:53.323 - INFO - Streamed logs:

    
    [notice] A new release of pip is available: 26.0.1 -> 26.2.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-08-27 08:55:56,968 INFO]: Looking for component `prepare_data` in --component_module_path `/tmp/tmp.xBEj8UKZ6X/ephemeral_component.py`
    [KFP Executor 2026-08-27 08:55:56,968 INFO]: Loading KFP component "prepare_data" from /tmp/tmp.xBEj8UKZ6X/ephemeral_component.py (directory "/tmp/tmp.xBEj8UKZ6X" and module name "ephemeral_component")
    [KFP Executor 2026-08-27 08:55:56,969 INFO]: Got executor_input:
    {
        "inputs": {
            "parameterValues": {
                "source": "dataset.csv"
            }
        },
        "outputs": {
            "artifacts": {
                "output_dataset": {
            

/home/ridwanfatur/miniconda3/envs/py3_12_9/lib/python3.12/site-packages/kfp/local/subprocess_task_handler.py:81: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image "pytorchlab/pytorch:2.4.1-cpu-py3.11-slim" in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the "DockerRunner" to run this task in a container.
  warnings.warn(


    
    [notice] A new release of pip is available: 26.0.1 -> 26.2.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-08-27 08:55:59,475 INFO]: Looking for component `train_model` in --component_module_path `/tmp/tmp.9sX6q2TRLO/ephemeral_component.py`
    [KFP Executor 2026-08-27 08:55:59,475 INFO]: Loading KFP component "train_model" from /tmp/tmp.9sX6q2TRLO/ephemeral_component.py (directory "/tmp/tmp.9sX6q2TRLO" and module name "ephemeral_component")
    [KFP Executor 2026-08-27 08:55:59,476 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "input_dataset": {
                    "artifacts": [
                        {
                            "name": "output_dataset",
                            "type": {
                                "schemaTitle": "system.Dataset",
                                "schemaVersion": "0.0.1"
                            },
                            "uri": "/home/ridwanfa